# Notebook 06 — Model Interpretability & ROI Simulator

**Phase 6 learning checkpoint.** We trained and saved a tuned XGBoost model in Phase 4. Now we open the black box: *why* does the model predict what it predicts, and *how much* is a renovation worth?

## What you will do here

1. Load the saved model and rebuild the feature matrix.
2. Compute SHAP values — a game-theoretic measure of each feature's contribution.
3. Global view: which features drive prices most, across all homes?
4. Local view: why did the model value *this specific home* at $X?
5. Dependence plots: how does the model's sensitivity to one feature change?
6. ROI simulator: what is a renovation worth in predicted dollars?

## Reading order

The module-level docstring in `src/interpretability.py` covers the SHAP theory. Read it alongside each section.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

from src.data_loader import load_ames, load_zillow_zhvi
from src.features import prepare_modeling_data
from src import models as M
from src import interpretability as I

print('Setup complete.')

## 1. Load model and data

We load the tuned XGBoost model saved in Phase 4 and reconstruct the same train/test split (same `random_state=42`) so we're explaining predictions on data the model never saw.

In [ ]:
model = M.load_model('xgb_ames_tuned')
print('Model loaded:', type(model.named_steps['model']).__name__)

X, y = prepare_modeling_data(load_ames(), zhvi=load_zillow_zhvi())
X_train, X_test, y_train, y_test = M.split(X, y)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features: {X.shape[1]}')

## 2. Build the SHAP explainer

``shap.TreeExplainer`` is a fast, exact algorithm for tree-based models. It computes, for each prediction, how much each feature contributed — starting from the average prediction (the *baseline*) and adding or subtracting each feature's share.

We pass the training set as the background distribution so SHAP knows what "average" means.

In [ ]:
print('Building SHAP explainer...')
explainer = I.get_explainer(model, X_train)
baseline = float(np.atleast_1d(explainer.expected_value)[0])
print(f'Baseline (average log-price): {baseline:.4f}')
print(f'Baseline (average price $):   ${np.expm1(baseline):,.0f}')

print('\nComputing SHAP values on test set...')
sv = I.shap_values(explainer, X_test)
print(f'SHAP values shape: {sv.shape}  (homes × features)')

## 3. Global feature importance

The bar chart shows **mean absolute SHAP value** per feature — the average size of each feature's impact on predictions, regardless of direction. Longer bar = more influential feature, on average across all test homes.

In [ ]:
I.plot_importance(sv, X_test, top_n=20)

## 4. SHAP summary (beeswarm) plot

Every dot is one home. The x-axis is the SHAP value (impact on log-price). Color shows whether the raw feature value was **high (red)** or **low (blue)**.

Patterns to look for:
- Red dots on the right → high feature value pushes price up (e.g. `total_sf`)
- Blue dots on the right → low feature value pushes price up (rare; sign of a complex interaction)
- Tightly clustered dots → the feature's effect is consistent across homes
- Wide spread → the feature's effect varies a lot (possibly an interaction)

In [ ]:
I.plot_summary(sv, X_test, max_display=20)

## 5. Local explanation — one home

Pick any home from the test set (change `home_idx` to explore different homes). The waterfall plot shows how that home's features push the prediction from the baseline up or down to the final predicted price.

Red bars push the price up. Blue bars push it down. The sum of all bars equals: `prediction − baseline`.

In [ ]:
home_idx = 0   # change this to explain a different home

pred_log  = model.predict(X_test.iloc[[home_idx]])[0]
pred_usd  = np.expm1(pred_log)
true_usd  = np.expm1(y_test.iloc[home_idx])

print(f'Home #{home_idx}')
print(f'  Predicted: ${pred_usd:,.0f}')
print(f'  Actual:    ${true_usd:,.0f}')
print(f'  Error:     ${abs(pred_usd - true_usd):,.0f}')

In [ ]:
I.plot_waterfall(explainer, X_test, idx=home_idx)

In [ ]:
# Same information as a table — handy for quick inspection.
I.explain_row(explainer, X_test, idx=home_idx)

## 6. Dependence plot — how one feature's effect varies

The dependence plot shows `total_sf` (x-axis) vs. its SHAP value (y-axis). Each dot is one home. Color shows an interaction feature chosen by SHAP.

- If the dots form a straight line → the model treats the feature linearly.
- If there's a curve → the model learned diminishing returns or a threshold.
- If the color bands separate vertically → there is an interaction (the effect of `total_sf` depends on the colored feature).

In [ ]:
I.plot_dependence(sv, X_test, feature='total_sf')

In [ ]:
# Try another feature — overall quality score is usually very revealing.
I.plot_dependence(sv, X_test, feature='overall_qual')

## 7. ROI Simulator

Now the practical question: **what is a renovation worth?**

We pick a home and simulate specific upgrades — changing one or more features to their "after renovation" values, re-running the model, and computing the price lift. Divide by estimated cost → ROI %.

> ⚠️ **Caveat.** The model was trained on historical sales. It captures what buyers *have paid* for these features, not what they *will pay* in the future. Use the output as a directional signal, not a precise appraisal.

In [ ]:
# Pick a home to upgrade — feel free to change this.
roi_idx = 5

home = X_test.iloc[roi_idx]
print(f'Home #{roi_idx} — selected feature values:')
print(f'  total_sf       : {home["total_sf"]:,.0f} sq ft')
print(f'  total_bath     : {home["total_bath"]:.1f} bathrooms')
print(f'  overall_qual   : {home["overall_qual"]:.0f} / 10')
print(f'  has_pool       : {home["has_pool"]:.0f}')
print(f'  garage_cars    : {home["garage_cars"]:.0f} car garage')
print(f'  Predicted price: ${I.predict_dollars(model, X_test.iloc[[roi_idx]])[0]:,.0f}')

In [ ]:
scenarios = [
    {
        'label':    'Add a full bathroom',
        'upgrades': {'total_bath': home['total_bath'] + 1},
        'cost':     15_000,
    },
    {
        'label':    'Finish 500 sq ft of basement',
        'upgrades': {'total_sf': home['total_sf'] + 500},
        'cost':     25_000,
    },
    {
        'label':    'Upgrade overall quality +1',
        'upgrades': {'overall_qual': min(home['overall_qual'] + 1, 10)},
        'cost':     40_000,
    },
    {
        'label':    'Expand garage to 2 cars',
        'upgrades': {'garage_cars': 2},
        'cost':     20_000,
    },
    {
        'label':    'Add pool',
        'upgrades': {'has_pool': 1},
        'cost':     50_000,
    },
]

rois = I.roi_table(model, X_test, roi_idx, scenarios)

display_df = rois.copy()
for col in ['before ($)', 'after ($)', 'price_lift ($)', 'cost ($)']:
    display_df[col] = display_df[col].apply(lambda x: f'${x:,.0f}')
display_df['roi_%'] = display_df['roi_%'].apply(lambda x: f'{x:.1f}%' if pd.notna(x) else 'N/A')
display_df

## 8. Combine upgrades

You can pass multiple features in one scenario to simulate a full renovation package.

In [ ]:
before, after, lift = I.simulate_upgrade(
    model, X_test, roi_idx,
    upgrades={
        'total_bath':   home['total_bath'] + 1,
        'total_sf':     home['total_sf'] + 500,
        'overall_qual': min(home['overall_qual'] + 1, 10),
    }
)
combined_cost = 15_000 + 25_000 + 40_000

print('Full renovation package:')
print(f'  Before : ${before:,.0f}')
print(f'  After  : ${after:,.0f}')
print(f'  Lift   : ${lift:,.0f}')
print(f'  Cost   : ${combined_cost:,.0f}')
print(f'  ROI    : {lift / combined_cost * 100:.1f}%')

## 9. Wrap-up — what did you learn?

Answer these before moving to Phase 7:

1. **SHAP vs. feature importance from the model.** XGBoost has a built-in `.feature_importances_` attribute. Why is SHAP more useful? What does SHAP tell you that the native importance metric does not?

2. **Reading the beeswarm.** Pick the top feature from the summary plot. Describe in plain English what the color and position pattern tells you about how the model uses that feature.

3. **Waterfall interpretation.** Look at the waterfall for `home_idx=0`. Which single feature had the biggest positive impact on price? Which had the biggest negative? Does that make intuitive sense given the feature values?

4. **Dependence plot non-linearity.** In the `total_sf` dependence plot, does the relationship look linear? If there is a bend, at roughly what square footage does it occur? What might explain that?

5. **ROI caveat.** The ROI simulator shows a bathroom addition has a certain return. Name two real-world reasons the actual return might be very different from the model's estimate.

6. **Diminishing returns.** If you run the quality upgrade scenario on a home that already has `Overall Qual = 9`, is the ROI likely higher or lower than on a home with `Overall Qual = 5`? Test it — does the model agree with your intuition?

When you can answer these, you are ready for **Phase 7: Power BI Export & Dashboard**.